Versão aprimorada do Software de Otimizaçção de Rotas!

In [ ]:
# Instalação da biblioteca DEAP
!pip install deap

In [ ]:
import random
import numpy as np
from deap import base, creator, tools, algorithms
import time

# Constantes
TAXA_MUTACAO_INICIAL = 0.1
ELITISMO_SIZE = 5
MAX_GERACOES_SEM_MELHORIA = 50
MAX_GERACOES_BASE = 100
TEMPO_MAXIMO = 60  # segundos
TAMANHO_POPULACAO_BASE = 100

# Pontos de entrega e matriz de distâncias corrigida
PONTOS = {
    1: "Almoxarifado",
    2: "Biblioteca",
    3: "Bloco 1 de Aulas",
    4: "Bloco 2 de Aulas",
    5: "Bloco administrativo",
    6: "Bloco 1 de Professores",
    7: "Bloco 2 de Professores",
    8: "Estacionamento",
    9: "Garagem",
    10: "Ginásio",
    11: "Laboratórios 1",
    12: "Laboratórios 2",
    13: "Memorial Paulo Freire",
    14: "Praça das Flores",
    15: "Restaurante Universitário",
    16: "Condomínio Canaã",
    17: "Residêncial Vermelho",
}
# Matriz de Distância 1

DISTANCIAS = np.array([
    # 1     2     3     4     5     6     7     8     9    10    11    12    13    14    15    16    17
    [0,   350,  550,  290,  500,  140,  290,  750,   50,  120, 1850,  400,  450,  170,  110, 1200, 1400],  # 1
    [350,    0,  400,  190,  190,  210,  190,  400,  350,  350,  150,  240,  340,  190,  250,  900, 1100],  # 2
    [550,  400,    0,  400,  400,  400,  400,  600,  550,  550, 1900,  160,  600,  400,  450, 1100, 1300],  # 3
    [290,  190,  400,    0,  350,  150,  130,  210,  290,  290,  200,  230,  190,  120,  180, 1100, 1300],  # 4
    [500,  190,  400,  350,    0,  350,  350,  550,  500,  500,  130,  230,  500,  350,  400,  750,  950],  # 5
    [140,  210,  400,  150,  350,    0,  150,  350,  150,  140,  230,  260,  300,  250,   34, 1100, 1300],  # 6
    [290,  190,  400,  130,  350,  150,    0,  200,  290,  280,  210,  240,  160,  120,  180, 1100, 1300],  # 7
    [750,  400,  600,  210,  550,  350,  200,    0,  500,  500,  400,  450,   56,  300,  400, 1300, 1500],  # 8
    [50,   350,  550,  290,  500,  150,  290,  500,    0,  120,  350,  400,  450,  170,   70, 1200, 1400],  # 9
    [120,  350,  550,  290,  500,  140,  280,  500,  120,    0,  350,  400,  450,  170,  110, 1200, 1400],  # 10
    [1850, 150, 1900,  200,  130,  230,  210,  400,  350,  350,    0,   95,  400,  200,  270,  850, 1100],  # 11
    [400,  240,  160,  230,  230,  260,  240,  450,  400,  400,   95,    0,  400,  240,  300, 1000, 1200],  # 12
    [450,  340,  600,  190,  500,  300,  160,   56,  450,  450,  400,  400,    0,  290,  350, 1200, 1400],  # 13
    [170,  190,  400,  120,  350,  250,  120,  300,  170,  170,  200,  240,  290,    0,   61, 1100, 1300],  # 14
    [110,  250,  450,  180,  400,   34,  180,  400,   70,  110,  270,  300,  350,   61,    0, 1100, 1300],  # 15
    [1200, 900, 1100, 1100,  750, 1100, 1100, 1300, 1200, 1200,  850, 1000, 1200, 1100, 1100,    0,  210],  # 16
    [1400,1100, 1300, 1300,  950, 1300, 1300, 1500, 1400, 1400, 1100, 1200, 1400, 1300, 1300,  210,    0]   # 17
])
def calcular_distancia_total(individuo, distancias, pontos_selecionados):
    """Calcula a distância total de um percurso, incluindo o retorno ao ponto inicial"""
    distancia_total = 0
    num_pontos = len(individuo)

    for i in range(num_pontos):
        idx1 = pontos_selecionados[individuo[i]] - 1  # Converte para índice da matriz
        idx2 = pontos_selecionados[individuo[(i + 1) % num_pontos]] - 1
        distancia_total += distancias[idx1][idx2]

    return distancia_total

def avaliar(individuo):
    """Função de avaliação do fitness (distância total)"""
    global pontos_selecionados, DISTANCIAS
    distancia_total = calcular_distancia_total(individuo, DISTANCIAS, pontos_selecionados)
    return (distancia_total,)

# Definição dos tipos do DEAP
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

def exibir_matriz_distancias(pontos_selecionados):
    """Exibe a matriz de distâncias para os pontos selecionados"""
    print("\nMatriz de Distâncias (em metros):")
    num_pontos = len(pontos_selecionados)
    tamanho_celula = 10

    # Cabeçalho
    print("┌" + "─" * tamanho_celula + ("┬" + "─" * tamanho_celula) * num_pontos + "┐")
    print("│" + f"{'':^{tamanho_celula}}", end="")
    for ponto in [PONTOS[p] for p in pontos_selecionados]:
        print(f"│{ponto[:tamanho_celula]:^{tamanho_celula}}", end="")
    print("│")
    print("├" + "─" * tamanho_celula + ("┼" + "─" * tamanho_celula) * num_pontos + "┤")

    # Linhas da matriz
    for i, ponto1 in enumerate(pontos_selecionados):
        print(f"│{PONTOS[ponto1][:tamanho_celula]:^{tamanho_celula}}", end="")
        for j, ponto2 in enumerate(pontos_selecionados):
            distancia = DISTANCIAS[ponto1 - 1][ponto2 - 1]
            print(f"│{distancia:^{tamanho_celula}}", end="")
        print("│")
        if i < num_pontos - 1:
            print("├" + "─" * tamanho_celula + ("┼" + "─" * tamanho_celula) * num_pontos + "┤")

    print("└" + "─" * tamanho_celula + ("┴" + "─" * tamanho_celula) * num_pontos + "┘")

def algoritmo_genetico(pontos_selecionados, taxa_mutacao_inicial=TAXA_MUTACAO_INICIAL,
                      elitismo_size=ELITISMO_SIZE, max_geracoes_sem_melhoria=MAX_GERACOES_SEM_MELHORIA,
                      tempo_maximo=TEMPO_MAXIMO):
    """Executa o algoritmo genético para encontrar a melhor rota"""
    num_pontos = len(pontos_selecionados)
    tamanho_populacao = min(TAMANHO_POPULACAO_BASE, 10 * num_pontos)

    toolbox = base.Toolbox()
    toolbox.register("indices", random.sample, range(num_pontos), num_pontos)
    toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.indices)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual, tamanho_populacao)
    toolbox.register("mate", tools.cxOrdered)
    toolbox.register("mutate", tools.mutShuffleIndexes, indpb=taxa_mutacao_inicial)
    toolbox.register("select", tools.selTournament, tournsize=3)
    toolbox.register("evaluate", avaliar)

    pop = toolbox.population()
    fitnesses = list(map(toolbox.evaluate, pop))
    for ind, fit in zip(pop, fitnesses):
        ind.fitness.values = fit

    best_individuals = tools.selBest(pop, elitismo_size)
    melhor_fitness = best_individuals[0].fitness.values[0]
    geracoes_sem_melhoria = 0

    inicio = time.time()

    for g in range(MAX_GERACOES_BASE):
        if time.time() - inicio > tempo_maximo:
            print(f"Tempo máximo de execução ({tempo_maximo}s) atingido.")
            break

        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))

        # Crossover
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < 0.7:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        # Mutação
        for mutant in offspring:
            if random.random() < taxa_mutacao_inicial:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        # Avaliação
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        # Elitismo
        pop[:] = offspring + best_individuals
        best_individuals = tools.selBest(pop, elitismo_size)

        # Verifica melhoria
        novo_melhor_fitness = best_individuals[0].fitness.values[0]
        if novo_melhor_fitness < melhor_fitness:
            melhor_fitness = novo_melhor_fitness
            geracoes_sem_melhoria = 0
        else:
            geracoes_sem_melhoria += 1

        if geracoes_sem_melhoria >= max_geracoes_sem_melhoria:
            print(f"Convergência atingida após {g + 1} gerações.")
            break

    return tools.selBest(pop, 1)[0]

def perguntar_tentar_novamente():
    """Pergunta ao usuário se deseja executar novamente"""
    while True:
        resposta = input("\nDeseja tentar novamente? (s/n): ").strip().lower()
        if resposta in ['s', 'n']:
            return resposta == 's'
        else:
            print("Por favor, digite 's' para sim ou 'n' para não.")

def main():
    """Função principal do programa"""
    global pontos_selecionados
    while True:
        print("\nBem-vindo ao Software de Otimização de Rotas com Algoritmos Genéticos!")
        print("Lista de Locais de Entrega Disponíveis:")
        for idx, ponto in PONTOS.items():
            print(f"{idx}: {ponto}")

        # Seleção de pontos pelo usuário
        while True:
            try:
                entrada = input("\nInforme os números dos locais de entrega desejados, separados por vírgula: ")
                pontos_selecionados = [int(idx) for idx in entrada.split(',')]
                pontos_selecionados = list(set(pontos_selecionados))  # Remove duplicatas

                if len(pontos_selecionados) < 2:
                    print("Erro: Você deve selecionar pelo menos 2 locais.")
                elif all(idx in PONTOS for idx in pontos_selecionados):
                    break
                else:
                    print("Erro: Algum número informado não corresponde a um local válido.")
            except ValueError:
                print("Erro: Digite apenas números separados por vírgula.")

        # Seleção do ponto de partida
        while True:
            try:
                ponto_partida = int(input("Informe o número do ponto de partida: "))
                if ponto_partida in pontos_selecionados:
                    break
                else:
                    print("Erro: O ponto de partida deve ser um dos pontos selecionados.")
            except ValueError:
                print("Erro: Digite apenas números.")

        # Exibir matriz de distâncias
        exibir_matriz_distancias(pontos_selecionados)

        # Executar algoritmo genético
        print("\nCalculando a melhor rota...")
        melhor_rota_indices = algoritmo_genetico(pontos_selecionados)
        melhor_rota_numeros = [pontos_selecionados[i] for i in melhor_rota_indices]

        # Reorganizar rota para começar no ponto de partida
        indice_partida = melhor_rota_numeros.index(ponto_partida)
        melhor_rota_ordenada = melhor_rota_numeros[indice_partida:] + melhor_rota_numeros[:indice_partida]

        # Exibir resultados
        print("\nMelhor Trajeto Calculado:")
        distancia_total = 0

        # Imprime a rota circular
        for i in range(len(melhor_rota_ordenada)):
            ponto_atual = melhor_rota_ordenada[i]
            proximo_ponto = melhor_rota_ordenada[(i + 1) % len(melhor_rota_ordenada)]
            distancia = DISTANCIAS[ponto_atual - 1][proximo_ponto - 1]
            distancia_total += distancia
            print(f"{i + 1}. {PONTOS[ponto_atual]} -> {PONTOS[proximo_ponto]} ({distancia} metros)")

        print(f"\nDistância total Percorrida: {distancia_total} metros")

        if not perguntar_tentar_novamente():
            print("\nObrigado por usar o Software de Otimização de Rotas. Até logo!")
            break

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nOperação cancelada pelo usuário.")
    except Exception as e:
        print(f"\nOcorreu um erro inesperado: {e}")